March Madness / college basketball prediction project centered on building a model that predicts NCAA tournament games from historical team-level data.

In [1]:
import numpy as np
import pandas as pd
import os

In [2]:
CURRENT_SEASON = 2026
DATA_DIR = 'march-machine-learning-mania-2026'

In [6]:
csv_files = [f for f in os.listdir(DATA_DIR) if f.endswith('.csv')]
data = {os.path.splitext(f)[0]: pd.read_csv(os.path.join(DATA_DIR, f)) for f in sorted(csv_files)}

for name, df in data.items():
    print(f"{name}: {df.shape}")

Cities: (509, 3)
Conferences: (51, 2)
MConferenceTourneyGames: (6793, 5)
MGameCities: (91940, 6)
MMasseyOrdinals: (5819228, 5)
MNCAATourneyCompactResults: (2585, 8)
MNCAATourneyDetailedResults: (1449, 34)
MNCAATourneySeedRoundSlots: (776, 5)
MNCAATourneySeeds: (2626, 3)
MNCAATourneySlots: (2586, 4)
MRegularSeasonCompactResults: (198079, 8)
MRegularSeasonDetailedResults: (124031, 34)
MSeasons: (42, 6)
MSecondaryTourneyCompactResults: (1865, 9)
MSecondaryTourneyTeams: (1895, 3)
MTeamCoaches: (13900, 5)
MTeamConferences: (13753, 3)
MTeamSpellings: (1178, 2)
MTeams: (381, 4)
SampleSubmissionStage1: (519144, 2)
SampleSubmissionStage2: (132133, 2)
WConferenceTourneyGames: (6481, 5)
WGameCities: (88621, 6)
WNCAATourneyCompactResults: (1717, 8)
WNCAATourneyDetailedResults: (961, 34)
WNCAATourneySeeds: (1744, 3)
WNCAATourneySlots: (1780, 4)
WRegularSeasonCompactResults: (142093, 8)
WRegularSeasonDetailedResults: (86773, 34)
WSeasons: (29, 6)
WSecondaryTourneyCompactResults: (906, 9)
WSecondaryT

In [7]:
display(data['MRegularSeasonDetailedResults'])

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,...,LFGA3,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF
0,2003,10,1104,68,1328,62,N,0,27,58,...,10,16,22,10,22,8,18,9,2,20
1,2003,10,1272,70,1393,63,N,0,26,62,...,24,9,20,20,25,7,12,8,6,16
2,2003,11,1266,73,1437,61,N,0,24,58,...,26,14,23,31,22,9,12,2,5,23
3,2003,11,1296,56,1457,50,N,0,18,38,...,22,8,15,17,20,9,19,4,3,23
4,2003,11,1400,77,1208,71,N,0,30,61,...,16,17,27,21,15,12,10,7,1,14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124026,2026,118,1373,76,1351,61,H,0,24,53,...,22,7,9,6,25,12,6,3,7,15
124027,2026,118,1378,90,1408,62,H,0,30,67,...,28,18,27,7,22,13,4,3,18,18
124028,2026,118,1389,63,1265,56,A,0,24,49,...,19,8,12,14,18,10,11,4,11,17
124029,2026,118,1455,84,1427,67,A,0,30,74,...,17,20,28,13,24,3,5,8,6,14


In [8]:
print(data['MRegularSeasonDetailedResults'].columns)

Index(['Season', 'DayNum', 'WTeamID', 'WScore', 'LTeamID', 'LScore', 'WLoc',
       'NumOT', 'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR',
       'WAst', 'WTO', 'WStl', 'WBlk', 'WPF', 'LFGM', 'LFGA', 'LFGM3', 'LFGA3',
       'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF'],
      dtype='str')


In [9]:
BOX_STATS = ['Score', 'FGM', 'FGA', 'FGM3', 'FGA3', 'FTM', 'FTA',
             'OR', 'DR', 'Ast', 'TO', 'Stl', 'Blk', 'PF']

results = data['MRegularSeasonDetailedResults'].sort_values(
    ['Season', 'DayNum']).reset_index(drop=True)

# {(season, team_id): {'sums': {stat: float}, 'game_count': int}}
team_rolling = {}

def get_or_create_entry(season, team_id):
    key = (season, team_id)
    if key not in team_rolling:
        sums = {}
        for stat in BOX_STATS:
            sums[stat] = 0.0
            sums[f'opponent_{stat}'] = 0.0
        team_rolling[key] = {'sums': sums, 'game_count': 0}
    return team_rolling[key]

def update_team(entry, game, own_prefix, opp_prefix):
    entry['game_count'] += 1
    for stat in BOX_STATS:
        entry['sums'][stat] += game[f'{own_prefix}{stat}']
        entry['sums'][f'opponent_{stat}'] += game[f'{opp_prefix}{stat}']

for _, game in results.iterrows():
    winner_entry = get_or_create_entry(game['Season'], game['WTeamID'])
    loser_entry = get_or_create_entry(game['Season'], game['LTeamID'])

    # TODO: featurize both teams' rolling stats here (pre-update)
    # for building training data later

    update_team(winner_entry, game, 'W', 'L')
    update_team(loser_entry, game, 'L', 'W')

print(f"Processed {len(results)} games, {len(team_rolling)} team-seasons")

Processed 124031 games, 8346 team-seasons


In [10]:
rows = []
for (season, team_id), entry in team_rolling.items():
    row = {'Season': season, 'TeamID': team_id, 'game_count': entry['game_count']}
    for stat_name, total in entry['sums'].items():
        row[f'avg_{stat_name.lower()}'] = total / entry['game_count']
    rows.append(row)

team_season_stats = pd.DataFrame(rows).sort_values(['Season', 'TeamID']).reset_index(drop=True)

# Rate stats (from season totals — properly weighted)
team_season_stats['fg_pct'] = team_season_stats['avg_fgm'] / team_season_stats['avg_fga']
team_season_stats['fg3_pct'] = team_season_stats['avg_fgm3'] / team_season_stats['avg_fga3']
team_season_stats['ft_pct'] = team_season_stats['avg_ftm'] / team_season_stats['avg_fta']
team_season_stats['fg3_rate'] = team_season_stats['avg_fga3'] / team_season_stats['avg_fga']

# Same for opponent (defensive) rates
team_season_stats['opponent_fg_pct'] = team_season_stats['avg_opponent_fgm'] / team_season_stats['avg_opponent_fga']
team_season_stats['opponent_fg3_pct'] = team_season_stats['avg_opponent_fgm3'] / team_season_stats['avg_opponent_fga3']
team_season_stats['opponent_ft_pct'] = team_season_stats['avg_opponent_ftm'] / team_season_stats['avg_opponent_fta']
team_season_stats['opponent_fg3_rate'] = team_season_stats['avg_opponent_fga3'] / team_season_stats['avg_opponent_fga']

display(team_season_stats)

,Season,TeamID,game_count,avg_score,avg_opponent_score,avg_fgm,avg_opponent_fgm,avg_fga,avg_opponent_fga,avg_fgm3,...,avg_pf,avg_opponent_pf,fg_pct,fg3_pct,ft_pct,fg3_rate,opponent_fg_pct,opponent_fg3_pct,opponent_ft_pct,opponent_fg3_rate
0,2003,1102,28,57.250000,57.000000,19.142857,19.285714,39.785714,42.428571,7.821429,...,18.750000,18.357143,0.481149,0.375643,0.651357,0.523339,0.454545,0.382184,0.710575,0.292929
1,2003,1103,27,78.777778,78.148148,27.148148,27.777778,55.851852,57.000000,5.444444,...,19.851852,22.444444,0.486074,0.338710,0.736390,0.287798,0.487329,0.362903,0.719064,0.322287
2,2003,1104,28,69.285714,65.000000,24.035714,23.250000,57.178571,55.500000,6.357143,...,18.035714,19.250000,0.420362,0.320144,0.709898,0.347283,0.418919,0.332090,0.708333,0.344916
3,2003,1105,26,71.769231,76.653846,24.384615,27.000000,61.615385,58.961538,7.576923,...,20.230769,19.076923,0.395755,0.364815,0.705986,0.337079,0.457926,0.357456,0.668760,0.297456
4,2003,1106,28,63.607143,63.750000,23.428571,21.714286,55.285714,53.392857,6.107143,...,18.178571,16.142857,0.423773,0.346154,0.646421,0.319121,0.406689,0.314554,0.707317,0.284950
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8341,2026,1477,29,66.931034,75.758621,23.344828,26.724138,56.000000,58.620690,8.034483,...,16.896552,15.034483,0.416872,0.310253,0.687379,0.462438,0.455882,0.373065,0.692833,0.380000
8342,2026,1478,30,72.366667,73.666667,24.733333,25.333333,53.800000,60.133333,7.566667,...,17.533333,17.866667,0.459727,0.344985,0.716511,0.407683,0.421286,0.358974,0.737374,0.389135
8343,2026,1479,29,68.551724,69.172414,25.896552,23.724138,58.206897,52.931034,5.793103,...,18.344828,15.344828,0.444905,0.316981,0.734411,0.313981,0.448208,0.332046,0.705701,0.337459
8344,2026,1480,28,75.142857,80.678571,27.678571,28.571429,62.607143,60.000000,6.892857,...,17.250000,17.035714,0.442099,0.335652,0.699612,0.328009,0.476190,0.335505,0.730645,0.365476
